# 03 - Analyse Descriptive des Données

Comparaison Lyon / Paris / Bordeaux — juin 2025.

**Objectif :** comprendre les données avant de modéliser.
- Quel est le prix médian dans chaque ville ?
- Quels types de logements sont les plus représentés ?
- Quelles variables sont corrélées au prix ?

## Chargement

`COLORS` : une couleur par ville pour des graphiques cohérents.
`os.makedirs(..., exist_ok=True)` : crée le dossier sans erreur s'il existe déjà.

In [ ]:
import pandas as pd, numpy as np, matplotlib, matplotlib.pyplot as plt, os
matplotlib.rcParams['figure.dpi'] = 100
os.makedirs('../reports/figures', exist_ok=True)

CITIES = {'Lyon':'lyon','Paris':'paris','Bordeaux':'bordeaux'}
COLORS = {'Lyon':'steelblue','Paris':'coral','Bordeaux':'mediumseagreen'}
dfs = {n: pd.read_csv(f'../data/processed/{k}_clean.csv') for n, k in CITIES.items()}
print({k: df.shape for k, df in dfs.items()})

## Statistiques descriptives

`describe()` : count, mean, std, min, Q1, médiane, Q3, max.
`display()` : affiche un tableau HTML dans Jupyter (plus lisible que print).

In [ ]:
for name, df in dfs.items():
    print(f'\n=== {name} ===')
    display(df[['price','accommodates','bedrooms','beds','bathrooms',
                'review_scores_rating','availability_365']].describe().round(2))

## Tableau comparatif

**Prix médian** plus robuste que la moyenne : les quelques villas à 2000€/nuit ne faussent pas la médiane.
**`replace(0, float('nan'))`** pour les notes : les 0 signifient 'pas encore noté', on les exclut.

In [ ]:
summary = {}
for name, df in dfs.items():
    summary[name] = {
        'Nb annonces'         : len(df),
        'Prix médian (€)'     : round(df['price'].median(), 1),
        'Prix moyen (€)'      : round(df['price'].mean(), 1),
        'Capacité méd. (p.)'  : df['accommodates'].median(),
        'Note moyenne'        : round(df['review_scores_rating'].replace(0,float('nan')).mean(),2),
        'Superhosts (%)'      : round(df['host_is_superhost'].mean()*100, 1),
        'Dispo moy. (j/an)'   : round(df['availability_365'].mean(), 0),
    }
display(pd.DataFrame(summary))

## Distribution des prix

On plafonne à 99e percentile pour le graphique : les quelques annonces à 5000€ écraseraient tout.
`axvline` : ligne verticale à la médiane pour repère visuel.
`plt.savefig(..., bbox_inches='tight')` : évite les coupures de titre ou d'axes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    cap = df['price'].quantile(0.99)
    data = df[df['price'] <= cap]['price']
    ax.hist(data, bins=60, color=COLORS[name], edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='black', linestyle='--',
               label=f'Médiane : {data.median():.0f}€')
    ax.set_title(f'Distribution prix - {name}')
    ax.set_xlabel('Prix / nuit (€)'); ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/prix_distribution.png', bbox_inches='tight')
plt.show()

## Prix médian par type de logement

`groupby('room_type')['price'].median()` : médiane du prix dans chaque groupe.
`barh` : barplot horizontal, plus lisible pour les étiquettes longues.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    med = df.groupby('room_type')['price'].median().sort_values()
    ax.barh(med.index, med.values, color=COLORS[name], alpha=0.85)
    ax.set_title(f'Prix par type - {name}'); ax.set_xlabel('Prix médian (€)')
    for i, v in enumerate(med.values): ax.text(v+1, i, f'{v:.0f}€', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../reports/figures/prix_par_type.png', bbox_inches='tight')
plt.show()

## Top 10 quartiers les plus chers

`nlargest(10)` : garde les 10 plus grandes valeurs.
`invert_yaxis()` : le quartier le plus cher apparaît en haut (lecture naturelle).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (name, df) in zip(axes, dfs.items()):
    top = df.groupby('neighbourhood_cleansed')['price'].median().nlargest(10)
    ax.barh(top.index, top.values, color=COLORS[name], alpha=0.85)
    ax.invert_yaxis(); ax.set_title(f'Top 10 quartiers - {name}')
    ax.set_xlabel('Prix médian (€)')
plt.tight_layout()
plt.savefig('../reports/figures/top_quartiers.png', bbox_inches='tight')
plt.show()

## Corrélation avec le prix

Coefficient de Pearson : entre -1 et +1.
+1 = si X augmente, prix augmente. 0 = aucune relation linéaire.

**Résultat attendu :** `accommodates` = corrélation la plus forte (~0.4–0.6).
C'est pourquoi on l'utilise comme seule variable en régression simple.

In [ ]:
for name, df in dfs.items():
    room_map = {'Entire home/apt':3,'Hotel room':2,'Private room':1,'Shared room':0}
    df2 = df.copy()
    df2['room_type_code'] = df['room_type'].map(room_map)
    num_cols = ['price','accommodates','bedrooms','beds','bathrooms','minimum_nights',
                'availability_365','review_scores_rating','host_is_superhost','room_type_code']
    cols = [c for c in num_cols if c in df2.columns]
    corr = df2[cols].corr()[['price']].sort_values('price', ascending=False)
    print(f'\n=== Corrélation avec le prix - {name} ===')
    display(corr.round(3))